# Localización Óptima de Centros de Distribución en Bolivia mediante Machine Learning

**Ejemplo didáctico — Clustering con K-Means ponderado por demanda**

En este notebook resolvemos un problema clásico de logística: **¿dónde ubicar los centros de distribución (CD) para atender la demanda de todo Bolivia?**

Metodología (similar al ejemplo visto en clase con Perú):

1. **Carga de datos**: 1,000 puntos de demanda (localidades) distribuidos en las 10 principales ciudades de Bolivia, cada uno con su demanda anual en pallets.
2. **Método del codo**: para estimar el número óptimo de clusters (K).
3. **K-Means ponderado**: la demanda entra como peso (`sample_weight`), de modo que los puntos con más demanda "jalan" más el centro del cluster.
4. **Distancias Haversine**: distancias reales en km sobre la superficie terrestre.
5. **Mapa interactivo** con Folium: clientes, clusters y centros de distribución.

> **Dataset**: `coordenadas_bolivia_1000_puntos.txt` (muestra estratificada del archivo maestro de 5,000 puntos).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import folium
import requests
from io import StringIO
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# =====================================================================================
# FUNCIÓN HAVERSINE (DISTANCIA EN KM)
# =====================================================================================
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calcula distancia Haversine en km entre:
    - un punto (lat1, lon1) y
    - uno o muchos puntos (lat2, lon2)
    """
    R = 6371.0  # radio promedio de la Tierra en km

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c  # km

## 1. Carga de datos desde GitHub

El dataset se lee directamente desde el repositorio del curso. Cada fila es una localidad con:

| Columna | Descripción |
|---|---|
| `LOCALIDAD` | Nombre de la localidad |
| `CIUDAD_BASE` | Ciudad principal a la que pertenece |
| `LATITUD`, `LONGITUD` | Coordenadas geográficas |
| `DEMANDA_PALLETS_ANUAL` | Demanda anual en pallets |
| `REGION`, `ZONA_GEOGRAFICA` | Clasificación geográfica (Oriente, Occidente, Altiplano, Valles, Llanos...) |

In [ ]:
# =====================================================================================
# 1. CARGA DE DATOS DESDE GITHUB
# =====================================================================================
GITHUB_URL = "https://raw.githubusercontent.com/santiagonajera/LOCALIZACION-OPTIMA-DE-CENTROS-DE-DISTRIBUCION-MEDIANTE-MACHINE-LEARNING/main/coordenadas_bolivia_1000_puntos.txt"

resp = requests.get(GITHUB_URL)
resp.raise_for_status()
df = pd.read_csv(StringIO(resp.text), sep="\t")

DEM_COL = "DEMANDA_PALLETS_ANUAL"

print(f"Total puntos de demanda: {len(df):,}")
print(f"Demanda total: {df[DEM_COL].sum():,} pallets/año")

print("\nPuntos por ciudad base:")
print(df["CIUDAD_BASE"].value_counts())

print("\nDemanda por región:")
print(df.groupby("REGION")[DEM_COL].sum().sort_values(ascending=False))

df.head()

## 2. Método del codo: ¿cuántos centros de distribución?

K-Means necesita que le digamos cuántos clusters (K) buscar. El **método del codo** grafica la *inercia* (suma de distancias cuadradas al centro del cluster) para distintos K: cuando agregar un cluster ya no reduce mucho la inercia, encontramos el "codo".

**Detalle clave**: usamos `sample_weight = demanda` para que la inercia también refleje la demanda, no solo la geografía.

In [ ]:
# =====================================================================================
# 2. PREPARACIÓN DE DATOS Y CÁLCULO DE K ÓPTIMO (MÉTODO DEL CODO)
# =====================================================================================
# Variables para clustering: latitud y longitud (escaladas)
X = df[["LATITUD", "LONGITUD"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# La demanda entra como PESO: los puntos con más pallets "jalan" más el centro
pesos = df[DEM_COL].values

inercias = []
k_range = range(2, 21)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled, sample_weight=pesos)
    inercias.append(kmeans.inertia_)

# Segunda derivada de la inercia para encontrar el "codo"
d1 = np.diff(inercias)
d2 = np.diff(d1)
k_optimo = int(np.argmax(d2) + 3)      # offset por las dos derivadas
k_sugerido = min(max(k_optimo, 4), 8)  # limitar entre 4 y 8 CDs

print(f"K óptimo matemático: {k_optimo}")
print(f"K sugerido (ajustado): {k_sugerido}")

# Gráfico del codo
plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inercias, "o-")
plt.axvline(k_sugerido, color="r", linestyle="--", label=f"K sugerido = {k_sugerido}")
plt.xlabel("K")
plt.ylabel("Inercia ponderada")
plt.title("Método del codo - Demanda en Bolivia")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. K-Means final y selección de los Centros de Distribución

Con el K sugerido:

1. Entrenamos K-Means sobre las coordenadas escaladas, ponderando por demanda.
2. Calculamos el **centro de gravedad ponderado** de cada cluster.
3. Elegimos como CD la **localidad real más cercana** a ese centro de gravedad (no podemos construir un almacén en medio de la nada).

In [ ]:
# =====================================================================================
# 3. K-MEANS FINAL Y CENTROS DE DISTRIBUCIÓN
# =====================================================================================
kmeans_final = KMeans(n_clusters=k_sugerido, random_state=42, n_init=20)
df["CLUSTER"] = kmeans_final.fit_predict(X_scaled, sample_weight=pesos)

nombres_centros = []
coordenadas_centros = []

for i in range(k_sugerido):
    cluster_data = df[df["CLUSTER"] == i]

    # Centro de gravedad ponderado por demanda del cluster
    lat_cg = np.average(cluster_data["LATITUD"], weights=cluster_data[DEM_COL])
    lon_cg = np.average(cluster_data["LONGITUD"], weights=cluster_data[DEM_COL])

    # CD = localidad real más cercana al centro de gravedad (distancia Haversine)
    dist = haversine_km(
        lat_cg, lon_cg,
        cluster_data["LATITUD"].values,
        cluster_data["LONGITUD"].values
    )
    loc_centro = cluster_data.iloc[dist.argmin()]

    nombres_centros.append(loc_centro["CIUDAD_BASE"])
    coordenadas_centros.append([loc_centro["LATITUD"], loc_centro["LONGITUD"]])

print("Centros de Distribución seleccionados:")
for i in range(k_sugerido):
    print(f"  CD #{i+1}: {nombres_centros[i]}  ({coordenadas_centros[i][0]:.4f}, {coordenadas_centros[i][1]:.4f})")

## 4. Resumen por Centro de Distribución

Tabla `df_resultados`: cuántas localidades atiende cada CD, demanda total y distancias promedio/máxima (Haversine, en km).

In [ ]:
# =====================================================================================
# 4. RESUMEN POR CD (df_resultados) - DISTANCIAS HAVERSINE
# =====================================================================================
resultados_clusters = []
for i in range(k_sugerido):
    cluster_data = df[df["CLUSTER"] == i]
    demanda_total = cluster_data[DEM_COL].sum()
    num_localidades = len(cluster_data)

    distancias = haversine_km(
        coordenadas_centros[i][0],
        coordenadas_centros[i][1],
        cluster_data["LATITUD"].values,
        cluster_data["LONGITUD"].values
    )

    resultados_clusters.append({
        "Cluster": i + 1,
        "Centro": nombres_centros[i],
        "Localidades": num_localidades,
        "Demanda_Total": demanda_total,
        "Demanda_Promedio": demanda_total / num_localidades,
        "Dist_Promedio_km": distancias.mean(),
        "Dist_Maxima_km": distancias.max(),
        "Latitud": coordenadas_centros[i][0],
        "Longitud": coordenadas_centros[i][1]
    })

df_resultados = pd.DataFrame(resultados_clusters)
display(df_resultados.round(2))

## 5. Tabla de asignaciones: quién atiende a quién

Tabla `df_asignaciones`: cada localidad con su CD asignado y la distancia aproximada en km.

In [ ]:
# =====================================================================================
# 5. TABLA "QUIÉN ATIENDE A QUIÉN" (df_asignaciones)
# =====================================================================================
df_asignaciones = df[[
    "LOCALIDAD", "CIUDAD_BASE", "LATITUD", "LONGITUD",
    DEM_COL, "TIPO_LOCALIDAD", "REGION",
    "ZONA_GEOGRAFICA", "CLUSTER"
]].copy()

df_asignaciones.rename(columns={DEM_COL: "DEMANDA"}, inplace=True)

df_asignaciones["CD_ASIGNADO"] = df_asignaciones["CLUSTER"].apply(lambda x: f"CD #{x+1}")
df_asignaciones["CENTRO_DISTRIBUCION"] = df_asignaciones["CLUSTER"].apply(lambda x: nombres_centros[x])

# Distancia Haversine de cada localidad a su CD
df_asignaciones["DISTANCIA_CD_KM"] = [
    haversine_km(row["LATITUD"], row["LONGITUD"],
                 coordenadas_centros[row["CLUSTER"]][0],
                 coordenadas_centros[row["CLUSTER"]][1])
    for _, row in df_asignaciones.iterrows()
]

df_asignaciones = df_asignaciones.sort_values(["CLUSTER", "DEMANDA"], ascending=[True, False])
display(df_asignaciones.head(15))

## 6. Mapa interactivo

- **Marcadores con ícono de casa** = Centros de Distribución.
- **Círculos de color** = localidades; el color indica el cluster/CD que las atiende y el tamaño indica la demanda.

Haz clic en cualquier punto para ver su detalle.

In [ ]:
# =====================================================================================
# 6. MAPA FOLIUM: CLIENTES + CENTROS + ASIGNACIÓN
# =====================================================================================
lat_centro = df["LATITUD"].mean()
lon_centro = df["LONGITUD"].mean()

mapa = folium.Map(location=[lat_centro, lon_centro], zoom_start=6, tiles="OpenStreetMap")

colores = ["red", "blue", "green", "purple", "orange", "darkred", "lightred", "beige"]

# Centros de distribución (marcadores con ícono de casa)
for i in range(k_sugerido):
    lat_cd, lon_cd = coordenadas_centros[i]
    nombre_cd = nombres_centros[i]

    folium.Marker(
        location=[lat_cd, lon_cd],
        popup=folium.Popup(f"CD #{i+1}<br>{nombre_cd}<br>({lat_cd:.4f}, {lon_cd:.4f})", max_width=250),
        icon=folium.Icon(color=colores[i], icon="home", prefix="fa"),
        tooltip=f"CD #{i+1} - {nombre_cd}"
    ).add_to(mapa)

# Clientes (todas las localidades)
for _, row in df_asignaciones.iterrows():
    cl = row["CLUSTER"]
    demanda = row["DEMANDA"]

    # tamaño del punto según demanda
    if demanda >= 5000:
        r = 7
    elif demanda >= 1000:
        r = 5
    else:
        r = 3

    folium.CircleMarker(
        location=[row["LATITUD"], row["LONGITUD"]],
        radius=r,
        color=colores[cl],
        fill=True,
        fill_color=colores[cl],
        fill_opacity=0.6,
        weight=1,
        popup=folium.Popup(
            f"<b>{row['LOCALIDAD']}</b><br>"
            f"Ciudad base: {row['CIUDAD_BASE']}<br>"
            f"Demanda: {demanda:,.0f} pallets/año<br>"
            f"Atendido por: {row['CD_ASIGNADO']} ({row['CENTRO_DISTRIBUCION']})<br>"
            f"Distancia aprox: {row['DISTANCIA_CD_KM']:.1f} km",
            max_width=250
        )
    ).add_to(mapa)

# Leyenda
leyenda_html = """
<div style="position: fixed;
     bottom: 10px; left: 10px; width: 260px; height: auto;
     background-color: white; z-index:9999; font-size:13px;
     border:2px solid grey; border-radius: 5px; padding: 10px">
     <b>Centros de Distribución (CD) - Bolivia</b><br>
     <i>Color = cluster / CD</i><br><br>
"""
for i in range(k_sugerido):
    leyenda_html += f"CD #{i+1} - {nombres_centros[i]}<br>"
leyenda_html += "</div>"

mapa.get_root().html.add_child(folium.Element(leyenda_html))

# Guardar el mapa como HTML y mostrarlo
mapa.save("mapa_bolivia_cds.html")
display(HTML(mapa._repr_html_()))

## 7. Ejercicios propuestos para los estudiantes

1. **Fija K = 5** manualmente (en lugar del K automático). ¿Qué nueva ciudad aparece como CD? ¿Mejora la distancia promedio?
2. **Repite con K = 6**. ¿Qué pasa con Trinidad y Cobija?
3. **Quita la ponderación** (`sample_weight=None`) y compara: ¿cómo cambian las ubicaciones de los CDs cuando la demanda no importa?
4. **Análisis por región**: usa las columnas `REGION` y `ZONA_GEOGRAFICA` para comparar la demanda del Altiplano vs. los Llanos. ¿Tiene sentido un CD en el norte (Cobija)?
5. **Función objetivo**: calcula el costo logístico total como `suma(demanda × distancia_km)` para K = 4, 5 y 6. ¿Cuál K minimiza el costo?
6. **Reto**: reemplaza K-Means por `KMedoids` (librería `scikit-learn-extra`). ¿Por qué un medoide puede ser mejor elección de CD que un centroide?